# Multi-Hop Research QA

## Answer Complex Questions Requiring Evidence From Multiple Documents

**Project 26** - Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Multi-hop question answering with reasoning chains |
| Method | Question decomposition + iterative retrieval + evidence fusion |
| Retrieval | Dense embedding search (sentence-transformers) |
| Corpus | 30-document multi-domain tech knowledge base |
| Evaluation | Hop-level recall, evidence completeness, single-hop vs multi-hop comparison |


## Project Overview

### The Problem: Questions That Need Multiple Documents

Most RAG systems retrieve once and answer. But many real questions
require **combining evidence from multiple documents**:

- *"Which cloud provider offers both the cheapest serverless option AND the best Kubernetes support?"*
  - Need pricing docs + orchestration docs + comparison
- *"How does the security model differ between the IaaS and PaaS tiers?"*
  - Need IaaS doc + PaaS doc + security doc

### Multi-Hop Reasoning Chain

```
Complex question
      |
      v
Decompose into sub-questions
      |
      v
For each sub-question:
  Retrieve relevant chunks
  Extract evidence
      |
      v
Fuse all evidence
      |
      v
Synthesize final answer
```

Each **hop** = one retrieval step for one sub-question.
Multi-hop = chaining multiple retrievals to gather complete evidence.


## Learning Goals

1. Understand **why** single-pass retrieval fails on complex questions
2. Implement question decomposition into sub-questions
3. Build an iterative retrieval chain (multi-hop)
4. Fuse evidence from multiple retrieval rounds
5. Compare single-hop vs multi-hop retrieval quality
6. Design reasoning chains for different question types


## Problem Statement

Given a complex question that spans multiple topics in the knowledge base:

1. **Decompose** the question into 2-4 simpler sub-questions
2. **Retrieve** evidence for each sub-question independently
3. **Fuse** all retrieved evidence into a comprehensive context
4. **Compare** against single-pass retrieval on the same questions

The multi-hop approach should retrieve evidence from more relevant
documents than single-pass, especially for cross-topic questions.


## Why Multi-Hop QA Matters

| Single-hop | Multi-hop |
|------------|----------|
| One retrieval per question | Multiple targeted retrievals |
| Misses cross-topic evidence | Covers all sub-topics |
| Retrieves by overall similarity | Retrieves by specific facets |
| Fast but incomplete | Slower but thorough |

**Real-world examples requiring multi-hop:**
- Research synthesis (gathering evidence across papers)
- Legal analysis (connecting statutes + case law + regulations)
- Technical troubleshooting (error + config + environment)
- Competitive analysis (product A features vs product B features)


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import re, random
from typing import List, Dict, Tuple, Set
from dataclasses import dataclass, field

import numpy as np
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
K_PER_HOP = 3      # chunks retrieved per sub-question
K_SINGLE  = 5      # chunks for single-pass baseline

print(f"Config: model={EMBEDDING_MODEL}")
print(f"  Multi-hop: K={K_PER_HOP} per sub-question")
print(f"  Single-hop baseline: K={K_SINGLE}")


## Knowledge Base

A 30-document multi-domain corpus spanning programming languages,
cloud platforms, databases, ML frameworks, and DevOps tools.
Multi-hop questions require bridging across these domains.


In [ ]:
corpus = [
    # -- Programming Languages --
    {"id": "PL01", "text": "Python is a general-purpose programming language created by Guido van Rossum in 1991. It emphasizes readability and supports multiple paradigms including procedural, object-oriented, and functional programming. Python is the most popular language for machine learning and data science.", "topic": "programming"},
    {"id": "PL02", "text": "Rust is a systems programming language developed by Mozilla, first released in 2015. It guarantees memory safety without garbage collection through its ownership system. Rust is used for performance-critical applications, web assembly, and embedded systems.", "topic": "programming"},
    {"id": "PL03", "text": "Go (Golang) was created by Google in 2009. It features built-in concurrency with goroutines and channels. Go compiles to native machine code and is widely used for cloud infrastructure, microservices, and CLI tools. Kubernetes and Docker are written in Go.", "topic": "programming"},
    {"id": "PL04", "text": "TypeScript is a strongly-typed superset of JavaScript developed by Microsoft in 2012. It adds static type checking, interfaces, and generics to JavaScript. TypeScript compiles to JavaScript and is widely used for large-scale web applications.", "topic": "programming"},
    {"id": "PL05", "text": "Julia is a high-performance programming language for technical computing, created in 2012. It combines the speed of C with the usability of Python. Julia uses multiple dispatch as its core paradigm and is popular in scientific computing and numerical analysis.", "topic": "programming"},

    # -- Cloud Platforms --
    {"id": "CP01", "text": "Amazon Web Services (AWS) is the largest cloud provider with over 200 services. Key services include EC2 for compute, S3 for storage, Lambda for serverless, and EKS for Kubernetes. AWS has the most data center regions globally with 33 regions.", "topic": "cloud"},
    {"id": "CP02", "text": "Microsoft Azure is the second-largest cloud provider. It integrates deeply with Microsoft enterprise products. Key services include Azure VMs, Blob Storage, Azure Functions for serverless, and AKS for Kubernetes. Azure has 60+ regions worldwide.", "topic": "cloud"},
    {"id": "CP03", "text": "Google Cloud Platform (GCP) is known for its data and AI services. Key services include Compute Engine, Cloud Storage, Cloud Functions for serverless, and GKE for Kubernetes. GCP offers BigQuery for data warehousing and Vertex AI for machine learning.", "topic": "cloud"},
    {"id": "CP04", "text": "Cloud serverless pricing varies significantly. AWS Lambda charges $0.20 per million requests plus $0.0000166667 per GB-second. Azure Functions free tier includes 1 million requests/month. Google Cloud Functions charges $0.40 per million invocations. All three offer generous free tiers.", "topic": "cloud"},
    {"id": "CP05", "text": "Managed Kubernetes comparison: AWS EKS costs $0.10/hour per cluster. Azure AKS has no cluster management fee. Google GKE Autopilot charges per pod resource usage. GKE is considered the most mature Kubernetes service. AKS offers the best Azure AD integration.", "topic": "cloud"},

    # -- Databases --
    {"id": "DB01", "text": "PostgreSQL is an open-source relational database known for extensibility and SQL compliance. It supports JSON, full-text search, and geospatial queries via PostGIS. PostgreSQL is ACID-compliant and handles both OLTP and analytical workloads.", "topic": "database"},
    {"id": "DB02", "text": "MongoDB is a document-oriented NoSQL database storing data in BSON format. It provides flexible schemas, horizontal scaling through sharding, and a rich query language. MongoDB Atlas is the managed cloud version available on all major cloud providers.", "topic": "database"},
    {"id": "DB03", "text": "Redis is an in-memory data structure store used as database, cache, and message broker. It supports strings, hashes, lists, sets, and sorted sets. Redis achieves sub-millisecond latency for read and write operations. Redis Cluster provides automatic sharding.", "topic": "database"},
    {"id": "DB04", "text": "Apache Cassandra is a distributed NoSQL database designed for high availability across multiple data centers. It uses a peer-to-peer architecture with no single point of failure. Cassandra is optimized for write-heavy workloads and time-series data.", "topic": "database"},
    {"id": "DB05", "text": "Neo4j is a graph database that stores data as nodes, relationships, and properties. It uses the Cypher query language. Neo4j excels at traversing connected data like social networks, recommendation engines, and knowledge graphs.", "topic": "database"},

    # -- ML Frameworks --
    {"id": "ML01", "text": "PyTorch is an open-source machine learning framework developed by Meta AI Research. It uses dynamic computational graphs and is the most popular framework for research. PyTorch supports GPU acceleration via CUDA and has a rich ecosystem including torchvision, torchaudio, and torchtext.", "topic": "ml"},
    {"id": "ML02", "text": "TensorFlow is an open-source ML framework developed by Google Brain. It uses static computational graphs (with eager mode available). TensorFlow Serving enables production model deployment. TensorFlow Lite targets mobile and edge devices.", "topic": "ml"},
    {"id": "ML03", "text": "scikit-learn is a Python machine learning library built on NumPy and SciPy. It provides implementations of classification, regression, clustering, and dimensionality reduction algorithms. scikit-learn is the standard library for traditional machine learning.", "topic": "ml"},
    {"id": "ML04", "text": "Hugging Face Transformers is an open-source NLP library providing thousands of pretrained models. It supports PyTorch, TensorFlow, and JAX backends. The library covers text classification, generation, translation, summarization, and question answering tasks.", "topic": "ml"},
    {"id": "ML05", "text": "MLflow is an open-source platform for managing the ML lifecycle. It provides experiment tracking, model registry, and model serving. MLflow supports any ML library and can deploy models to various production environments including Docker and Kubernetes.", "topic": "ml"},

    # -- DevOps & Infrastructure --
    {"id": "DO01", "text": "Docker containers package applications with dependencies into portable units. Docker Hub hosts millions of container images. Docker Compose orchestrates multi-container applications locally. Docker uses layered filesystem for efficient image storage.", "topic": "devops"},
    {"id": "DO02", "text": "Kubernetes (K8s) orchestrates containers at scale. Core concepts include Pods, Services, Deployments, and StatefulSets. Kubernetes handles automatic scaling, rolling updates, and self-healing. It has become the standard for container orchestration.", "topic": "devops"},
    {"id": "DO03", "text": "Terraform by HashiCorp is an Infrastructure as Code tool using HCL (HashiCorp Configuration Language). It supports all major cloud providers and on-premises infrastructure. Terraform state tracks resource lifecycle. Modules enable reusable infrastructure patterns.", "topic": "devops"},
    {"id": "DO04", "text": "GitHub Actions is a CI/CD platform integrated with GitHub repositories. Workflows are defined in YAML files triggered by events like push, pull request, or schedule. Actions marketplace provides thousands of reusable workflow steps. Matrix builds test across multiple environments.", "topic": "devops"},
    {"id": "DO05", "text": "Prometheus is an open-source monitoring system with a time-series database. It collects metrics via pull-based scraping. PromQL query language enables flexible alerting rules. Prometheus integrates with Grafana for visualization and is the standard monitoring stack for Kubernetes.", "topic": "devops"},

    # -- Cross-domain connections --
    {"id": "XD01", "text": "Python ML ecosystem on AWS: SageMaker supports PyTorch and TensorFlow training. Lambda supports Python runtimes for inference. S3 stores training datasets. EC2 GPU instances (P4d, P5) provide NVIDIA A100 and H100 GPUs for training.", "topic": "cross"},
    {"id": "XD02", "text": "Kubernetes ML operations: Kubeflow provides ML pipelines on Kubernetes. MLflow can be deployed on K8s for experiment tracking. GPU scheduling in Kubernetes uses the NVIDIA device plugin. Model serving uses KServe (formerly KFServing) or Seldon.", "topic": "cross"},
    {"id": "XD03", "text": "Database selection for ML applications: PostgreSQL stores structured experiment metadata. MongoDB stores model artifacts and configurations. Redis caches model predictions for low-latency serving. Neo4j stores feature relationship graphs for feature engineering.", "topic": "cross"},
]

print(f"Knowledge base: {len(corpus)} documents")
topics = {}
for doc in corpus:
    topics[doc["topic"]] = topics.get(doc["topic"], 0) + 1
for topic, count in sorted(topics.items()):
    print(f"  {topic}: {count} documents")


## Dense Retriever

In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

doc_texts = [doc["text"] for doc in corpus]
doc_embeddings = encoder.encode(doc_texts, convert_to_numpy=True, show_progress_bar=False)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)


def retrieve(query: str, k: int = K_PER_HOP) -> List[Dict]:
    """Retrieve top-k documents by cosine similarity."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores = (doc_embeddings @ q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"id": corpus[i]["id"], "text": corpus[i]["text"],
             "topic": corpus[i]["topic"], "score": float(scores[i])}
            for i in top_idx]


test = retrieve("Kubernetes container orchestration", k=3)
print(f"\nRetriever ready (index: {doc_embeddings.shape})")
for r in test:
    print(f"  [{r["id"]}] {r["topic"]:>12s} score={r["score"]:.3f} | {r["text"][:55]}...")


## Question Decomposition

Break a complex question into simpler sub-questions.

In production, an LLM decomposes questions. Here we use rule-based
decomposition that detects common multi-hop patterns:

| Pattern | Example | Sub-questions |
|---------|---------|---------------|
| Comparison | "Compare X and Y" | "What is X?" + "What is Y?" |
| Bridge | "What does X use for Y?" | "What is X?" + "What Y options exist?" |
| Multi-fact | "Which X has Y and Z?" | "Which X has Y?" + "Which X has Z?" |


In [ ]:
def decompose_question(question: str) -> List[str]:
    """Decompose a complex question into sub-questions.
    
    Uses pattern matching to detect multi-hop needs.
    In production, an LLM would do this more flexibly.
    """
    q_lower = question.lower()
    sub_questions = []
    
    # Pattern 1: Comparison ("compare X and Y", "difference between X and Y")
    comp_match = re.search(
        r"(?:compare|difference|differ|vs|versus).*?\b(\w[\w\s]{2,30}?)\b.*?(?:and|vs|versus|with)\s+(\w[\w\s]{2,30})",
        q_lower
    )
    if comp_match:
        a, b = comp_match.group(1).strip(), comp_match.group(2).strip()
        sub_questions.append(f"What is {a} and what are its key features?")
        sub_questions.append(f"What is {b} and what are its key features?")
        # Add comparison facet if mentioned
        for facet in ["pricing", "cost", "performance", "security", "kubernetes", "serverless"]:
            if facet in q_lower:
                sub_questions.append(f"What is the {facet} of {a} and {b}?")
                break
        return sub_questions
    
    # Pattern 2: "and" connector (multi-topic)
    and_parts = re.split(r"\band\b|\balso\b|\bplus\b", question, flags=re.IGNORECASE)
    and_parts = [p.strip().rstrip("?. ") for p in and_parts if len(p.strip()) > 15]
    if len(and_parts) >= 2:
        for part in and_parts:
            if not part.endswith("?"):
                part = part + "?"
            sub_questions.append(part)
        return sub_questions
    
    # Pattern 3: Bridge questions (relationship between entities)
    bridge_patterns = [
        (r"which.*?(?:language|framework|tool|database).*?(?:used|use|support|for).*?(\w[\w\s]+)",
         lambda m: [f"What programming languages and frameworks exist?",
                    f"Which tools support {m.group(1).strip()}?"]),
        (r"how.*?deploy.*?(\w+).*?(?:to|on|using)\s+(\w[\w\s]+)",
         lambda m: [f"What is {m.group(1).strip()}?",
                    f"How to deploy on {m.group(2).strip()}?",
                    f"What deployment tools are available?"]),
    ]
    for pattern, gen_fn in bridge_patterns:
        m = re.search(pattern, q_lower)
        if m:
            return gen_fn(m)
    
    # Pattern 4: Extract key noun phrases as separate retrieval queries
    topic_keywords = {
        "programming": ["python", "rust", "go", "golang", "typescript", "julia", "language"],
        "cloud": ["aws", "azure", "gcp", "google cloud", "serverless", "lambda", "cloud"],
        "database": ["postgresql", "mongodb", "redis", "cassandra", "neo4j", "database", "sql"],
        "ml": ["pytorch", "tensorflow", "scikit", "hugging face", "mlflow", "machine learning", "ml"],
        "devops": ["docker", "kubernetes", "terraform", "github actions", "prometheus", "devops", "ci/cd"],
    }
    matched_topics = []
    for topic, keywords in topic_keywords.items():
        if any(kw in q_lower for kw in keywords):
            matched_topics.append(topic)
    
    if len(matched_topics) >= 2:
        for topic in matched_topics:
            sub_questions.append(f"{question.rstrip('?')} focusing on {topic}?")
        return sub_questions
    
    # Fallback: return original question + a broader version
    sub_questions.append(question)
    return sub_questions


# Demo
demo_questions = [
    "Compare AWS and Azure Kubernetes offerings and their pricing",
    "Which programming language is best for ML and what databases work well with it?",
    "How do I deploy a PyTorch model to Kubernetes?",
]

print("Question Decomposition Demo")
print("=" * 60)
for q in demo_questions:
    subs = decompose_question(q)
    print(f"\n  Q: {q}")
    for i, sq in enumerate(subs, 1):
        print(f"    Sub-{i}: {sq}")


## Multi-Hop Retriever

For each sub-question, retrieve K documents independently,
then merge all retrieved documents (deduplicating by ID).

This is the core of multi-hop retrieval: chain multiple
independent retrievals to build comprehensive evidence.


In [ ]:
@dataclass
class HopResult:
    """Result from one hop (one sub-question retrieval)."""
    sub_question: str
    hop_number: int
    retrieved_docs: List[Dict]
    doc_ids: List[str]
    topics_covered: Set[str]


@dataclass
class MultiHopResult:
    """Result from the full multi-hop pipeline."""
    original_question: str
    sub_questions: List[str]
    hops: List[HopResult]
    all_doc_ids: List[str]       # unique docs across all hops
    all_topics: Set[str]         # topics covered across all hops
    total_docs: int
    evidence_text: str           # merged evidence for answering


def multi_hop_retrieve(question: str, k_per_hop: int = K_PER_HOP) -> MultiHopResult:
    """Decompose question and retrieve evidence in multiple hops."""
    sub_questions = decompose_question(question)
    
    hops = []
    seen_ids = set()
    all_docs = []
    all_topics = set()
    
    for i, sq in enumerate(sub_questions, 1):
        docs = retrieve(sq, k=k_per_hop)
        hop_ids = [d["id"] for d in docs]
        hop_topics = set(d["topic"] for d in docs)
        
        hops.append(HopResult(
            sub_question=sq,
            hop_number=i,
            retrieved_docs=docs,
            doc_ids=hop_ids,
            topics_covered=hop_topics,
        ))
        
        # Deduplicate across hops
        for d in docs:
            if d["id"] not in seen_ids:
                seen_ids.add(d["id"])
                all_docs.append(d)
                all_topics.add(d["topic"])
    
    # Build merged evidence
    evidence = " ".join(d["text"] for d in all_docs)
    
    return MultiHopResult(
        original_question=question,
        sub_questions=sub_questions,
        hops=hops,
        all_doc_ids=[d["id"] for d in all_docs],
        all_topics=all_topics,
        total_docs=len(all_docs),
        evidence_text=evidence,
    )


# Single-hop baseline
def single_hop_retrieve(question: str, k: int = K_SINGLE) -> MultiHopResult:
    """Single-pass retrieval baseline."""
    docs = retrieve(question, k=k)
    return MultiHopResult(
        original_question=question,
        sub_questions=[question],
        hops=[HopResult(
            sub_question=question, hop_number=1,
            retrieved_docs=docs,
            doc_ids=[d["id"] for d in docs],
            topics_covered=set(d["topic"] for d in docs),
        )],
        all_doc_ids=[d["id"] for d in docs],
        all_topics=set(d["topic"] for d in docs),
        total_docs=len(docs),
        evidence_text=" ".join(d["text"] for d in docs),
    )


print("Multi-hop and single-hop retrievers defined.")


## Test Questions

Complex questions that require evidence from multiple documents
across different topics. Each has ground-truth relevant document IDs.


In [ ]:
test_questions = [
    {
        "question": "Compare AWS and GCP Kubernetes offerings and their pricing",
        "relevant": ["CP01", "CP03", "CP05"],
        "type": "comparison",
        "hops_needed": 2,
        "note": "Needs cloud platforms + Kubernetes pricing docs",
    },
    {
        "question": "Which programming language is best for machine learning and what databases work well with ML?",
        "relevant": ["PL01", "ML01", "ML03", "XD03"],
        "type": "bridge",
        "hops_needed": 2,
        "note": "Needs programming lang + ML framework + DB-for-ML docs",
    },
    {
        "question": "How do I deploy a PyTorch model to Kubernetes and what monitoring tools should I use?",
        "relevant": ["ML01", "DO02", "XD02", "DO05"],
        "type": "bridge",
        "hops_needed": 3,
        "note": "Needs PyTorch + K8s + ML ops + monitoring docs",
    },
    {
        "question": "Compare PostgreSQL and MongoDB for storing ML experiment data and which cloud providers offer managed versions?",
        "relevant": ["DB01", "DB02", "XD03", "CP01", "CP02", "CP03"],
        "type": "multi-fact",
        "hops_needed": 3,
        "note": "Needs PostgreSQL doc + MongoDB doc + DB-for-ML + cloud provider docs",
    },
    {
        "question": "What CI/CD tools integrate with Kubernetes and how do I monitor deployments?",
        "relevant": ["DO04", "DO02", "DO05", "XD02"],
        "type": "bridge",
        "hops_needed": 2,
        "note": "Needs CI/CD + K8s + monitoring docs",
    },
    {
        "question": "Which Google products are used in both cloud computing and machine learning?",
        "relevant": ["CP03", "ML02", "PL03"],
        "type": "bridge",
        "hops_needed": 2,
        "note": "Needs GCP services + Google ML framework + Go language docs",
    },
    {
        "question": "Compare the serverless offerings across all three major cloud providers and their pricing",
        "relevant": ["CP01", "CP02", "CP03", "CP04"],
        "type": "comparison",
        "hops_needed": 2,
        "note": "Needs AWS + Azure + GCP + pricing docs",
    },
    {
        "question": "What infrastructure as code tools work with Kubernetes and how do they integrate with CI/CD?",
        "relevant": ["DO03", "DO02", "DO04", "DO01"],
        "type": "bridge",
        "hops_needed": 2,
        "note": "Needs Terraform + K8s + GitHub Actions + Docker docs",
    },
]

print(f"Test questions: {len(test_questions)}")
for i, q in enumerate(test_questions, 1):
    print(f"  Q{i} [{q["type"]:>11s}] {len(q["relevant"])} docs needed | {q["question"][:55]}...")


## Evaluation Metrics

| Metric | What it measures |
|--------|------------------|
| **Evidence recall** | Fraction of ground-truth docs retrieved |
| **Topic coverage** | Number of distinct topics covered |
| **Unique docs** | Total unique documents retrieved |
| **Efficiency** | Recall per retrieval call |


In [ ]:
def evaluate_result(result: MultiHopResult, relevant_ids: List[str]) -> Dict:
    """Evaluate a retrieval result against ground truth."""
    retrieved_set = set(result.all_doc_ids)
    relevant_set = set(relevant_ids)
    
    hits = retrieved_set & relevant_set
    recall = len(hits) / len(relevant_set) if relevant_set else 0.0
    precision = len(hits) / len(retrieved_set) if retrieved_set else 0.0
    
    return {
        "recall": recall,
        "precision": precision,
        "hits": sorted(hits),
        "missed": sorted(relevant_set - retrieved_set),
        "total_retrieved": len(retrieved_set),
        "topics_covered": len(result.all_topics),
        "num_hops": len(result.hops),
    }


print("Evaluation metrics defined.")


## Benchmark: Single-Hop vs Multi-Hop

Run both approaches on all test questions and compare.


In [ ]:
results_single = []
results_multi = []

for tq in test_questions:
    q = tq["question"]
    relevant = tq["relevant"]
    
    # Single-hop
    sr = single_hop_retrieve(q, k=K_SINGLE)
    se = evaluate_result(sr, relevant)
    results_single.append({"question": q, "result": sr, "eval": se, "type": tq["type"]})
    
    # Multi-hop
    mr = multi_hop_retrieve(q, k_per_hop=K_PER_HOP)
    me = evaluate_result(mr, relevant)
    results_multi.append({"question": q, "result": mr, "eval": me, "type": tq["type"]})

print(f"Benchmark complete: {len(test_questions)} questions x 2 methods")


## Aggregate Results

In [ ]:
print(f"{'Metric':<20} {'Single-Hop':>12} {'Multi-Hop':>12} {'Delta':>8}")
print("-" * 54)

for metric_name, key in [("Recall", "recall"), ("Precision", "precision"),
                         ("Total Docs", "total_retrieved"), ("Topics", "topics_covered")]:
    s_avg = np.mean([r["eval"][key] for r in results_single])
    m_avg = np.mean([r["eval"][key] for r in results_multi])
    delta = m_avg - s_avg
    sign = "+" if delta >= 0 else ""
    print(f"{metric_name:<20} {s_avg:>12.3f} {m_avg:>12.3f} {sign}{delta:>7.3f}")

# Recall improvement
s_recall = np.mean([r["eval"]["recall"] for r in results_single])
m_recall = np.mean([r["eval"]["recall"] for r in results_multi])
pct = ((m_recall - s_recall) / s_recall * 100) if s_recall > 0 else 0
print(f"\nMulti-hop recall improvement: {pct:+.1f}%")


## Per-Question Detail

In [ ]:
for i, (sq, mq) in enumerate(zip(results_single, results_multi), 1):
    tq = test_questions[i - 1]
    se, me = sq["eval"], mq["eval"]
    
    s_better = se["recall"] > me["recall"]
    m_better = me["recall"] > se["recall"]
    winner = "MULTI-HOP" if m_better else ("SINGLE" if s_better else "TIE")
    
    print(f"\nQ{i} [{tq['type']:>11s}]: {tq['question'][:60]}...")
    print(f"  Relevant docs: {tq['relevant']}")
    print(f"  {'Method':<14} {'Recall':>7} {'Docs':>5} {'Topics':>7} {'Hits':>15} {'Missed':>15}")
    print(f"  {'-'*66}")
    print(f"  {'Single-hop':<14} {se['recall']:>7.2f} {se['total_retrieved']:>5} {se['topics_covered']:>7} {str(se['hits']):>15} {str(se['missed']):>15}")
    print(f"  {'Multi-hop':<14} {me['recall']:>7.2f} {me['total_retrieved']:>5} {me['topics_covered']:>7} {str(me['hits']):>15} {str(me['missed']):>15}")
    print(f"  Winner: {winner}")


## Multi-Hop Retrieval Traces

See how each hop contributes new evidence.
Good decomposition = each hop retrieves different relevant documents.


In [ ]:
for i, mq in enumerate(results_multi, 1):
    result = mq["result"]
    tq = test_questions[i - 1]
    
    print(f"\nQ{i}: {tq['question'][:65]}...")
    print(f"  Decomposed into {len(result.sub_questions)} sub-questions:")
    
    cumulative_ids = set()
    for hop in result.hops:
        new_ids = set(hop.doc_ids) - cumulative_ids
        cumulative_ids.update(hop.doc_ids)
        relevant_hits = set(hop.doc_ids) & set(tq["relevant"])
        
        print(f"\n    Hop {hop.hop_number}: {hop.sub_question[:55]}...")
        print(f"      Retrieved: {hop.doc_ids} (topics: {sorted(hop.topics_covered)})")
        print(f"      New docs: {sorted(new_ids)} | Relevant hits: {sorted(relevant_hits)}")
    
    print(f"\n    Total unique docs: {result.total_docs} | Topics: {sorted(result.all_topics)}")


## Results by Question Type

In [ ]:
for qtype in ["comparison", "bridge", "multi-fact"]:
    s_items = [r for r in results_single if r["type"] == qtype]
    m_items = [r for r in results_multi if r["type"] == qtype]
    if not s_items:
        continue
    s_recall = np.mean([r["eval"]["recall"] for r in s_items])
    m_recall = np.mean([r["eval"]["recall"] for r in m_items])
    s_topics = np.mean([r["eval"]["topics_covered"] for r in s_items])
    m_topics = np.mean([r["eval"]["topics_covered"] for r in m_items])
    
    print(f"\n{qtype.upper()} questions ({len(s_items)}):")
    print(f"  Single-hop: recall={s_recall:.3f}, topics={s_topics:.1f}")
    print(f"  Multi-hop:  recall={m_recall:.3f}, topics={m_topics:.1f}")
    delta = m_recall - s_recall
    print(f"  Delta: {delta:+.3f} recall")
    if delta > 0.05:
        print(f"  -> Multi-hop significantly helps for {qtype} questions")
    elif delta > -0.05:
        print(f"  -> Similar performance; multi-hop adds topic diversity")
    else:
        print(f"  -> Single-hop sufficient for this type")


## Error Analysis

When does multi-hop retrieval fail?


In [ ]:
print("Error Analysis\n")

# Where multi-hop failed
for i, (sq, mq) in enumerate(zip(results_single, results_multi), 1):
    tq = test_questions[i - 1]
    me = mq["eval"]
    
    if me["missed"]:
        print(f"  Q{i}: Multi-hop missed {me['missed']}")
        print(f"    Question: {tq['question'][:60]}...")
        print(f"    Decomposition: {mq['result'].sub_questions}")
        # Diagnose
        for missed_id in me["missed"]:
            missed_doc = next(d for d in corpus if d["id"] == missed_id)
            print(f"    Missing [{missed_id}] ({missed_doc['topic']}): {missed_doc['text'][:60]}...")
            # Check if any sub-question would have retrieved it with larger K
            for sq_text in mq["result"].sub_questions:
                docs_k10 = retrieve(sq_text, k=10)
                ranks = [j+1 for j, d in enumerate(docs_k10) if d["id"] == missed_id]
                if ranks:
                    sq_short = sq_text[:50]
                    print(f"      -> Would be at rank {ranks[0]} for sub-q: {sq_short}...  (K={K_PER_HOP} too small)")
        print()

# Where multi-hop hurt
print("\nCases where single-hop beat multi-hop:")
worse_count = 0
for i, (sq, mq) in enumerate(zip(results_single, results_multi), 1):
    if sq["eval"]["recall"] > mq["eval"]["recall"]:
        worse_count += 1
        print(f"  Q{i}: single recall={sq['eval']['recall']:.2f} > multi recall={mq['eval']['recall']:.2f}")
if worse_count == 0:
    print("  None -- multi-hop was never worse than single-hop.")


## Reasoning Chain Design

A reasoning chain determines **how** sub-questions feed into each other.

### Chain Types

| Chain Type | How it works | Example |
|-----------|-------------|----------|
| **Parallel** | All sub-questions retrieved independently | "Compare X and Y" |
| **Sequential** | Each hop uses results from the previous hop | "What does X use? How does that integrate with Y?" |
| **Tree** | Sub-questions branch and merge | Complex research synthesis |

Our implementation uses **parallel** chains (each sub-question retrieved
independently). Sequential chains are more powerful but require the
decomposer to know what hop 1 returned before generating hop 2's query.

### When to Use Each

- **Parallel:** Comparison, multi-fact, listing queries
- **Sequential:** Bridge queries where hop 2 depends on hop 1's answer
- **Tree:** Research synthesis requiring hierarchical evidence gathering


## Limitations

1. **Rule-based decomposition.** An LLM decomposer would handle
   diverse question patterns much better.

2. **Parallel chains only.** Our implementation can't do sequential
   chains where hop 2 depends on hop 1's results.

3. **No answer generation.** We measure retrieval quality but don't
   generate actual answers.

4. **Fixed K per hop.** Adaptive K would retrieve more for ambiguous
   sub-questions and less for specific ones.

5. **No evidence fusion quality.** Simply concatenating evidence
   doesn't ensure coherence.

6. **Small corpus.** 30 documents make multi-hop less critical;
   with 10,000+ documents, multi-hop becomes essential.


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| Too many sub-questions | Retrieves noise, dilutes evidence | Cap at 2-4 sub-questions |
| Not deduplicating across hops | Same doc counted multiple times | Track seen IDs |
| Ignoring decomposition quality | Bad sub-questions = bad retrieval | Evaluate decomposition separately |
| Using multi-hop for simple questions | Adds latency without benefit | Classify question complexity first |
| No fallback to single-hop | Multi-hop can fail on some patterns | Compare and choose best result |
| Sequential chains without context | Hop 2 query doesn't use hop 1 results | Pass hop 1 evidence to hop 2 decomposer |


## Mini Challenge

1. **LLM-based decomposition.** Replace rule-based decomposer with
   an LLM. Compare sub-question quality.

2. **Sequential chains.** Implement a chain where hop 2's query is
   informed by hop 1's retrieved documents.

3. **Adaptive complexity classifier.** Build a classifier that
   decides whether a question needs multi-hop or single-hop.

4. **Combine with compression.** Use Project 25's compression on
   the multi-hop evidence before generating an answer.

5. **Evidence graph.** Build a graph connecting documents by shared
   entities/topics. Use graph traversal for multi-hop instead of
   independent retrievals.


## Production Considerations

| Aspect | Approach |
|--------|----------|
| **Latency** | Multi-hop adds N x retrieval latency; parallelize sub-question retrieval |
| **Complexity routing** | Classify question complexity to decide single vs multi-hop |
| **Decomposition quality** | Use LLM with examples for reliable decomposition |
| **Evidence coherence** | Rerank merged evidence by relevance to original question |
| **Cost** | More retrieval calls + longer context = higher cost |
| **Caching** | Cache sub-question results; many questions share sub-queries |
| **Monitoring** | Track hop count, decomposition quality, and recall per hop |


## How to Improve This Project

1. **LLM decomposer** -- More robust question decomposition.
2. **Sequential chains** -- Hop 2 conditioned on hop 1 results.
3. **Entity linking** -- Use named entity recognition to guide sub-queries.
4. **Cross-encoder reranking** -- Rerank merged evidence.
5. **Answer generation** -- Add an LLM to synthesize the final answer.
6. **Complexity classifier** -- Route simple questions to single-hop.
7. **Larger corpus** -- Test with 1000+ documents.


## Key Takeaways

1. **Complex questions need multi-hop retrieval.** Single-pass retrieval
   cannot gather evidence scattered across multiple documents.

2. **Decomposition is the key step.** Good sub-questions lead to good
   retrieval. Bad sub-questions make multi-hop worse than single-hop.

3. **Parallel chains work for comparison/multi-fact questions.**
   Sequential chains are needed for bridge questions where later
   hops depend on earlier results.

4. **Multi-hop improves topic coverage.** Even when recall is similar,
   multi-hop retrieves from more distinct topic areas.

5. **Not always needed.** Simple questions don't benefit from multi-hop.
   Route by complexity to avoid unnecessary latency.

6. **Composable with other techniques.** Multi-hop (P26) + compression
   (P25) + citation verification (P24) + query rewriting (P22) =
   production-grade complex QA pipeline.

7. **Always compare against single-hop baseline.** Multi-hop adds
   complexity; prove it's worth it with metrics.
